### Import Committee Reports from Congress API

In [1]:
# Inital Setup
import numpy as np
import pandas as pd
import requests
import json
import dotenv
import os
import yaml
import llm
import pprint

from lxml import etree
from lxml import html
from bs4 import BeautifulSoup

In [2]:
botname = 'targ'
version = '0.0'
email = 'kve5hd@virginia.edu'
useragent =f'{botname}/{version} ({email}) python-requests/{requests.__version__}'
headers = {'User-Agent':useragent}
headers

{'User-Agent': 'targ/0.0 (kve5hd@virginia.edu) python-requests/2.32.5'}

In [3]:
root = "https://api.congress.gov//v3"
dotenv.load_dotenv()
congresskey = os.getenv('congresskey')

In [4]:
params = {'format': 'json',
          'api_key': congresskey}

In [5]:
endpoint = f'/committee-report'

r = requests.get(root + endpoint,
                 params=params,
                 headers=headers)
r

<Response [200]>

In [6]:
myjson = r.json()

In [7]:
myjson.keys()

dict_keys(['pagination', 'reports', 'request'])

In [8]:
myjson['reports']

[{'chamber': 'House',
  'citation': 'H. Rept. 119-1',
  'cmte_rpt_id': 289187,
  'congress': 119,
  'number': 1,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:12:39Z',
  'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/1?format=json'},
 {'chamber': 'House',
  'citation': 'H. Rept. 119-2',
  'cmte_rpt_id': 289189,
  'congress': 119,
  'number': 2,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:12:41Z',
  'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/2?format=json'},
 {'chamber': 'House',
  'citation': 'H. Rept. 119-3',
  'cmte_rpt_id': 289190,
  'congress': 119,
  'number': 3,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:12:42Z',
  'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/3?format=json'},
 {'chamber': 'House',
  'citation': 'H. Rept. 119-4',
  'cmte_rpt_id': 289191,
  'congress': 119,
  'number': 4,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:13:32Z',
  'url': 'https://api

In [9]:
cmt_reports = pd.json_normalize (myjson, record_path = ['reports'])

In [10]:
cmt_reports.columns

Index(['chamber', 'citation', 'cmte_rpt_id', 'congress', 'number', 'part',
       'type', 'updateDate', 'url'],
      dtype='str')

In [11]:
print(cmt_reports.iloc[0].to_dict())

{'chamber': 'House', 'citation': 'H. Rept. 119-1', 'cmte_rpt_id': 289187, 'congress': 119, 'number': 1, 'part': 1, 'type': 'HRPT', 'updateDate': '2025-05-27T14:12:39Z', 'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/1?format=json'}


In [12]:
row = cmt_reports.iloc[0]
row

chamber                                                    House
citation                                          H. Rept. 119-1
cmte_rpt_id                                               289187
congress                                                     119
number                                                         1
part                                                           1
type                                                        HRPT
updateDate                                  2025-05-27T14:12:39Z
url            https://api.congress.gov/v3/committee-report/1...
Name: 0, dtype: object

In [13]:
r = requests.get(
    f"{root}/{endpoint}/{row['congress']}/{row['type']}/{row['number']}/text",
    headers=headers,
    params=params)

pprint.pprint(r.json())

{'pagination': {'count': 2},
 'request': {'congress': '119',
             'contentType': 'application/json',
             'format': 'json',
             'reportNumber': '1',
             'reportType': 'hrpt'},
 'text': [{'formats': [{'isErrata': 'N',
                        'type': 'Formatted Text',
                        'url': 'https://www.congress.gov/119/crpt/hrpt1/generated/CRPT-119hrpt1.htm'}]},
          {'formats': [{'isErrata': 'N',
                        'type': 'PDF',
                        'url': 'https://www.congress.gov/119/crpt/hrpt1/CRPT-119hrpt1.pdf'}]}]}


In [14]:
htm_url = r.json()['text'][0]['formats'][0]['url']
response = requests.get(htm_url)
response.status_code

200

In [15]:
# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# Extract the text
text = soup.get_text()
pprint.pprint(text)

('\n'
 '\n'
 '119th Congress   }                                      {       Report\n'
 '                        HOUSE OF REPRESENTATIVES\n'
 ' 1st Session     }                                      {        119-1\n'
 '\n'
 '======================================================================\n'
 '\n'
 '\n'
 ' \n'
 ' PROVIDING FOR CONSIDERATION OF THE BILL (H.R. 471) TO EXPEDITE UNDER \n'
 '   THE NATIONAL ENVIRONMENTAL POLICY ACT OF 1969 AND IMPROVE FOREST \n'
 'MANAGEMENT ACTIVITIES ON NATIONAL FOREST SYSTEM LANDS, ON PUBLIC LANDS \n'
 'UNDER THE JURISDICTION OF THE BUREAU OF LAND MANAGEMENT, AND ON TRIBAL \n'
 'LANDS TO RETURN RESILIENCE TO OVERGROWN, FIRE-PRONE FORESTED LANDS, AND \n'
 'FOR OTHER PURPOSES, AND PROVIDING FOR CONSIDERATION OF THE BILL (S. 5) \n'
 '  TO REQUIRE THE SECRETARY OF HOMELAND SECURITY TO TAKE INTO CUSTODY \n'
 ' ALIENS WHO HAVE BEEN CHARGED IN THE UNITED STATES WITH THEFT, AND FOR \n'
 '                             OTHER PURPOSES\n'
 '\n'
 '             

In [16]:
pprint.pprint(r.json())

{'pagination': {'count': 2},
 'request': {'congress': '119',
             'contentType': 'application/json',
             'format': 'json',
             'reportNumber': '1',
             'reportType': 'hrpt'},
 'text': [{'formats': [{'isErrata': 'N',
                        'type': 'Formatted Text',
                        'url': 'https://www.congress.gov/119/crpt/hrpt1/generated/CRPT-119hrpt1.htm'}]},
          {'formats': [{'isErrata': 'N',
                        'type': 'PDF',
                        'url': 'https://www.congress.gov/119/crpt/hrpt1/CRPT-119hrpt1.pdf'}]}]}


## Test 1: Collect committee reports from 117th to 119th

In [17]:
congresses = [115, 116, 117, 118, 119]

all_reports = []

for congress in congresses:
    offset = 0
    while True:
        params = {
            'format': 'json',
            'api_key': congresskey,
            'congress': congress,
            'offset': offset,
            'limit': 250
        }
        
        r = requests.get (root + endpoint, 
                          params=params, 
                          headers=headers
        )
        
        reports = r.json().get('reports', [])
        if not reports:
            break
        
        all_reports.extend(reports)
        offset += 250
        
        if offset >= r.json().get('pagination', {}).get('count', 0):
            break

df = pd.json_normalize(all_reports)    

KeyboardInterrupt: 

In [ ]:
df

,chamber,citation,cmte_rpt_id,congress,number,part,type,updateDate,url
0,House,H. Rept. 119-1,289187,119,1,1,HRPT,2025-05-27T14:12:39Z,https://api.congress.gov/v3/committee-report/1...
1,House,H. Rept. 119-2,289189,119,2,1,HRPT,2025-05-27T14:12:41Z,https://api.congress.gov/v3/committee-report/1...
2,House,H. Rept. 119-3,289190,119,3,1,HRPT,2025-05-27T14:12:42Z,https://api.congress.gov/v3/committee-report/1...
3,House,H. Rept. 119-4,289191,119,4,1,HRPT,2025-05-27T14:13:32Z,https://api.congress.gov/v3/committee-report/1...
4,House,H. Rept. 119-5,289193,119,5,1,HRPT,2025-05-27T14:12:45Z,https://api.congress.gov/v3/committee-report/1...
...,...,...,...,...,...,...,...,...,...
99935,House,H. Rept. 104-651,193801,104,651,1,HRPT,2025-04-07T15:27:21Z,https://api.congress.gov/v3/committee-report/1...
99936,Senate,S. Rept. 104-284,193802,104,284,1,SRPT,2025-04-07T15:30:16Z,https://api.congress.gov/v3/committee-report/1...
99937,Senate,S. Rept. 104-259,193803,104,259,1,SRPT,2025-04-07T15:28:43Z,https://api.congress.gov/v3/committee-report/1...
99938,Senate,Ex. Rept. 104-6,220643,104,6,1,ERPT,2025-04-07T15:13:58Z,https://api.congress.gov/v3/committee-report/1...


In [ ]:
cmt_reports = df.to_csv('../data/cmt_reports.csv', index=False)

## Store committee reports in MongoDB

In [ ]:
import requests, time, logging
from bs4 import BeautifulSoup
from pymongo import MongoClient, errors
from datetime import datetime

In [ ]:
# Set up logging
logging.basicConfig(
    filename='cmt_reports_scraper.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [ ]:
client = MongoClient()
db = client['cmt_reports']
collection = db['cmt_reports']  

In [ ]:
# Function to fetch and parse HTML content

def fetch_text(row, headers, params):
    try:
        # Step1: text endpoint
        r = requests.get(
            f"{root}/{endpoint}/{row['congress']}/{row['type']}/{row['number']}/text",
            headers=headers,
            params={'api_key': congresskey,
                    'format': 'json'},
            timeout=10
        )
        
        r.raise_for_status()
        
        formats = r.json().get('text', [{}])[0].get('formats', [{}])
        htm = next((f for f in formats if f.get('type') == 'HTM'), None)
        
        if not htm_url:
            logging.warning(f"No HTM URL: {row['cmte_rpt_id']}")
            return None
        
        # Step 2: fetch HTML
        time.sleep(0.5)
        r = requests.get(htm_url, timeout=10)
        r.raise_for_status()
        
        text = BeautifulSoup(r.text, 'html_parser').get_text()
        return text
    
    except Exception as e:
        logging.error(f"Failed {row['cmte_rpt_id']}: {e}")
        return None


In [ ]:
# Main loop

for i, row in df.iterrows():
    if collection.find_one({'cmte_rpt_id': row['cmte_rpt_id']}):
        continue
    
    text = fetch_text(row)
    
    doc = {
        'cmte_rpt_id': row['cmte_rpt_id'],
        'congress': row['congress'],
        'type': row['type'],
        'number': row['number'],
        'part': row['part'],
        'chamber': row['chamber'],
        'citation': row['citation'],
        'updateDate': row['updateDate'],
        'raw_text': text,
        'fetched_at': datetime.utcnow(),
        'fetch_success': text is not None
    }

    try:
        collection.insert_one(doc)
        
    except errors.DuplicateKeyError:
        pass
    
    if i % 500 == 0:
        logging.info(f"Progress: {i}/{len(df)}")
        time.sleep(1)
        
print("Done")

TypeError: fetch_text() missing 2 required positional arguments: 'headers' and 'params'